In [ ]:
import numpy as np                                                                 # Operaciones numéricas (aunque aquí casi no se usa)
import random                                                                      # Generación de números aleatorios

"""
Nota sobre coordenadas:

Las coordenadas geográficas se almacenan como (lat, lon),
pero matemáticamente corresponden a (y, x).

Por esta razón, en las visualizaciones se trabaja con:
    - latitud  → eje Y
    - longitud → eje X

Esto implica que los cortes y particiones del KD-Tree se realizan
primero sobre Y y luego sobre X, aunque la tupla esté en formato (lat, lon).
"""

data = []

# Límites del Valle de Aburrá
LAT_MIN = 5.98                                                                     # Sur (más allá de Caldas) latitud en grados al norte del ecuador
LAT_MAX = 6.51                                                                     # Norte (Barbosa) latitud en grados al norte del ecuador
LON_MIN = -75.71                                                                   # Occidente, longitud en grados al oeste de Greenwich
LON_MAX = -75.21                                                                   # Oriente, longitud en grados al oeste de Greenwich

def generate_data():
    for i in range(10000):
        lat = random.uniform(LAT_MIN, LAT_MAX)
        lon = random.uniform(LON_MIN, LON_MAX)
        data.append((lat, lon))

generate_data()

In [ ]:
import random                                                                      # Generación de números aleatorios
import numpy as np                                                                 # Operaciones numéricas (aunque aquí casi no se usa)

# ------------------------------ Lista ligada ------------------------------------

# Nodo de la lista ligada
class Node:
    def __init__(self, coord):                                                     # Constructor del nodo
        self.coord = coord                                                         # Coordenada almacenada (lat, lon)
        self.next = None                                                           # Referencia al siguiente nodo (enlace)

# Lista ligada
class LinkedList:                                                                  # Estructura de lista simplemente enlazada
    def __init__(self):                                                            # Constructor de la lista
        self.head = None                                                           # Primer nodo de la lista (inicio)
        self.size = 0                                                              # Cantidad de elementos en la lista

    def append(self, coord):                                                       # Inserta un nuevo nodo al final
        new_node = Node(coord)                                                     # Crear nodo con la coordenada

        if not self.head:                                                          # Si la lista está vacía
            self.head = new_node                                                   # El nuevo nodo es el primero
        else:
            current = self.head
            while current.next:                                                    # Recorrer hasta el último nodo
                current = current.next
            current.next = new_node                                                # Enlazar el nuevo nodo al final

        self.size += 1                                                             # Incrementar tamaño de la lista

    def __repr__(self):                                                            # Representación en texto de la lista
        nodes, current = [], self.head
        while current:
            nodes.append(str(current.coord))                                       # Guardar coordenadas como string
            current = current.next
        return " -> ".join(nodes[:5]) + f" ... ({self.size} nodos)"                # Mostrar primeros 5 nodos

# ------------------------------ Lista ligada ------------------------------------




# --------------------------------- Árbol KD -------------------------------------

class Node_KD_2D:                                                                  # Clase para el nodo del árbol
    def __init__(self, value, left=None, right=None):                              # Constructor del nodo
        self.value = value                                                         # Punto almacenado (lat, lon)
        self.left = left                                                           # Subárbol izquierdo (valores menores)
        self.right = right                                                         # Subárbol derecho (valores mayores)

# --------------------------------- Árbol KD -------------------------------------




# ------------------------------- Funciones --------------------------------------

# ------------------------------ Fuerza Bruta ------------------------------------

def F_bruta_list(data):                                                           # Fuerza Bruta para listas
    sorted_by_x = sorted(data, key=lambda coord: coord[1])                        # Ordenar puntos por X (longitud → índice 1)
    sorted_by_y = sorted(data, key=lambda coord: coord[0])                        # Ordenar puntos por Y (latitud → índice 0)

    # Construir lista ligada ordenada por X
    list_x = LinkedList()                                                         # Crear lista ligada vacía
    for coord in sorted_by_x:                                                     # Recorrer lista de puntos
        list_x.append(coord)                                                      # Agregar punto a la lista ligada

    # Construir lista ligada ordenada por Y
    list_y = LinkedList()                                                         # Crear lista ligada vacía
    for coord in sorted_by_y:                                                     # Recorrer lista de puntos
        list_y.append(coord)                                                      # Agregar punto a la lista ligada

    return list_x, list_y                                                         # Retorna ambas listas ordenadas

# ------------------------------ Fuerza Bruta ------------------------------------


# -------------------------------- KD Tree ---------------------------------------

def KD_Tree(data, depth=0, k_dimension=2):                                        # Funcion para construir el KD Tree
    """
    Construye un KD-Tree a partir de una lista de puntos.

    data: lista de coordenadas (lat, lon)
    depth: profundidad actual del árbol (controla el eje de corte)
    k_dimension: número de dimensiones (2 en este caso)
    """

    if len(data) == 0:                                                            # Caso base: lista vacía
        return None

    if len(data) == 1:                                                            # Caso base: hoja
        return Node_KD_2D(data[0])

    axis = depth % k_dimension                                                    # Alterna eje: 0 → lat, 1 → lon
    other = (axis + 1) % k_dimension                                              # Eje secundario (para desempate)

    sorted_data = sorted(data, key=lambda coord: (coord[axis], coord[other]))     # Ordenar por el eje actual, usando el otro eje como criterio secundario

    mid = len(sorted_data) // 2                                                   # Índice del punto medio (mediana) (OJO: SOLO DATOS HOMOGENEOS, para distribuciones no homogeneas se implementa otro metodo)

    node = Node_KD_2D(sorted_data[mid])                                           # Nodo raíz en esta subdivisión

    # Construcción recursiva de subárboles
    node.left  = KD_Tree(sorted_data[:mid], depth + 1)                            # Subárbol izquierdo (menores)
    node.right = KD_Tree(sorted_data[mid + 1:], depth + 1)                        # Subárbol derecho (mayores)

    return node                                                                   # Retorna el nodo construido


def get_max_depth(node, depth=0):
    """
    Calcula la profundidad máxima del KD-Tree.
    """

    if node is None:                                                              # Si no hay nodo
        return depth - 1                                                          # Ajuste porque se pasó un nivel

    # Retorna la mayor profundidad entre ambos subárboles
    return max(get_max_depth(node.left, depth+1),
               get_max_depth(node.right, depth+1))

# -------------------------------- KD Tree ---------------------------------------